# VERA — RL Fine-Tuning (REINFORCE)

Runs **Phase 2 RL training** starting from an SFT checkpoint, using `rl_trainer_vera.py`.

**Loss**: REINFORCE + value baseline + entropy bonus + KL penalty vs frozen BC anchor  
**Expected time**: ~2–4 hrs for 100 epochs on T4 GPU (Language-Table)

---

## Before running
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your SFT checkpoint (`best_sft_vera.pt`) to Drive at:
   ```
   MyDrive/VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt
   ```
   *(rename the file from your seed123 SFT run — it goes in the **rl_seed123** folder)*
3. Run all cells top-to-bottom

## Output
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/best_rl_vera.pt` — best checkpoint by return
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/rl_vera_log.json` — full epoch log
- `MyDrive/VERA_LT_Checkpoints/rl_seed123/rl/sample_efficiency.csv` — for plotting
- Live epoch-by-epoch output in cell 6 below

## Running multiple seeds
Open separate Colab tabs and change `SEED` in Cell 4 (e.g. `SEED = 42`, `SEED = 456`).  
Each seed needs its own SFT checkpoint folder:  
`rl_seed42/best_sft_vera.pt`, `rl_seed456/best_sft_vera.pt`

In [1]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# ── Cell 2: Clone repo (or use Drive copy) ─────────────────────────────────
import os, sys

REPO_DRIVE = '/content/drive/MyDrive/VLA-Robot-Learning'
REPO_LOCAL = '/content/RLConditionedVLA'   # matches GitHub repo name

if os.path.isdir(REPO_DRIVE):
    print(f'Repo found on Drive at {REPO_DRIVE}')
    REPO = REPO_DRIVE
elif os.path.isdir(REPO_LOCAL):
    print(f'Repo already cloned at {REPO_LOCAL}')
    REPO = REPO_LOCAL
else:
    print('Cloning repo from GitHub...')
    os.system(f'git clone https://github.com/sara-kaz/RLConditionedVLA.git {REPO_LOCAL}')
    REPO = REPO_LOCAL
    print(f'Cloned to {REPO}')

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'Using repo: {REPO}')
print(f'Config path: {REPO}/configs/config.yaml')

In [ ]:
# ── Cell 3: Install dependencies ───────────────────────────────────────────
import subprocess, sys

def pip(*pkgs, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if no_deps:
        cmd.append('--no-deps')
    cmd.extend(pkgs)
    subprocess.check_call(cmd)

# Base deps
pip('ftfy', 'regex', 'tqdm', 'pyyaml', 'pillow', 'numpy')

# gym version compatible with language-table
pip('gym<=0.23.0')
pip('pybullet')

# language-table must be installed --no-deps from GitHub (not on PyPI)
pip('git+https://github.com/google-research/language-table.git', no_deps=True)

# CLIP
try:
    import clip
    print('CLIP already installed.')
except ImportError:
    pip('git+https://github.com/openai/CLIP.git')

# Verify
from language_table.environments import language_table as _lt_check
print('language_table import OK')
print('All dependencies installed.')

In [ ]:
# ── Cell 4: ⚙️  USER CONFIG ────────────────────────────────────────────────
import os, glob

MYDRIVE = '/content/drive/MyDrive'
SEED    = 123   # ← change to 42 or 456 for other seeds

# ── RL hyperparameters ─────────────────────────────────────────────────────
RL_EPOCHS          = 100
RL_NUM_ROLLOUTS    = 8
RL_MAX_EP_STEPS    = 60
RL_LR              = 3e-5
RL_ENTROPY_COEF    = 0.05
RL_KL_COEF         = 0.20
RL_VF_COEF         = 0.5
RL_GAMMA           = 0.99
RL_GRAD_CLIP       = 1.0
FREEZE_CLIP          = True
UNFREEZE_CLIP_VISION = False

# ── Language-Table data path ───────────────────────────────────────────────
LT_DATA_PATH = f'{MYDRIVE}/VERA_LT_Real/lt_real_data'

# ── Check DAgger checkpoint paths (from your v6 training code) ─────────────
DAGGER_CKPT_DIR = f'{MYDRIVE}/VERA_LT_Real/checkpoints/lt_full_vera_rl/seed{SEED}'
CANDIDATES = [
    f'{DAGGER_CKPT_DIR}/best_v3.pt',
    f'{DAGGER_CKPT_DIR}/latest_v6.pt',
]

print(f'Checking known DAgger checkpoint paths for seed {SEED}:\n')
SFT_CKPT = ''
for path in CANDIDATES:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✓ FOUND  {path}  ({size_mb:.0f} MB)')
        if not SFT_CKPT:
            SFT_CKPT = path
    else:
        print(f'  ✗ missing {path}')

if not SFT_CKPT:
    print('\nNot found in expected locations — scanning all of Drive for .pt files …')
    all_pts = glob.glob(f'{MYDRIVE}/**/*.pt', recursive=True)
    if all_pts:
        print(f'  All .pt files on Drive:')
        for p in sorted(all_pts):
            print(f'    {os.path.getsize(p)/1e6:6.0f} MB  {p}')
    else:
        print('  ✗ No .pt files found anywhere on Drive.')

# SFT_CKPT = f'{MYDRIVE}/path/to/override.pt'   # ← uncomment to override

# ── Output dir ────────────────────────────────────────────────────────────
RL_OUT_DIR = f'{DAGGER_CKPT_DIR}/rl'
os.makedirs(RL_OUT_DIR, exist_ok=True)

# ── Summary ───────────────────────────────────────────────────────────────
print(f'\nSFT_CKPT   = "{SFT_CKPT}"')
print(f'RL_OUT_DIR = "{RL_OUT_DIR}"')
print(f'LT_DATA    = "{LT_DATA_PATH}"  →  ', end='')
print('✓ found' if os.path.isdir(LT_DATA_PATH) else '✗ NOT FOUND — check path')
if SFT_CKPT:
    print('✓ Checkpoint found — proceed to Cell 5')

In [ ]:
# ── Cell 5: Write config + convert checkpoint to RL trainer format ──────────
import yaml, os, torch

if not SFT_CKPT:
    raise ValueError(
        'SFT_CKPT is empty — run Cell 4 first and resolve the missing checkpoint.'
    )
if not os.path.exists(SFT_CKPT):
    raise FileNotFoundError(f'Checkpoint not found: {SFT_CKPT}')

# ── Load DAgger checkpoint and re-save in RL trainer format ───────────────
# DAgger saves:  {'model': state_dict, 'pos_head': ..., 'epoch': ..., 'best_sr': ...}
# RL trainer expects: {'model_state': state_dict}   (line 375 of rl_trainer_vera.py)
print(f'Loading checkpoint: {SFT_CKPT}')
raw = torch.load(SFT_CKPT, map_location='cpu')

if isinstance(raw, dict):
    keys = list(raw.keys())
    print(f'  Checkpoint keys: {keys}')
    if 'model_state' in raw:
        model_sd = raw['model_state']
        print('  Format: RL trainer (model_state) — no conversion needed')
    elif 'model' in raw:
        model_sd = raw['model']
        sr = raw.get('best_sr', 0.0)
        ep = raw.get('epoch', '?')
        print(f'  Format: DAgger (model) — converting.  epoch={ep}, best_sr={sr:.1%}')
    else:
        model_sd = raw
        print('  Format: bare state dict')
else:
    raise TypeError(f'Unexpected checkpoint type: {type(raw)}')

SFT_LINK = f'{RL_OUT_DIR}/best_sft_vera.pt'
torch.save({'model_state': model_sd}, SFT_LINK)
print(f'  Saved converted checkpoint → {SFT_LINK}')

# ── Write patched config ───────────────────────────────────────────────────
CONFIG_PATH = f'{REPO}/configs/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['rl']['epochs']            = RL_EPOCHS
cfg['rl']['num_rollouts']      = RL_NUM_ROLLOUTS
cfg['rl']['max_episode_steps'] = RL_MAX_EP_STEPS
cfg['rl']['lr']                = float(RL_LR)
cfg['rl']['entropy_coef']      = float(RL_ENTROPY_COEF)
cfg['rl']['kl_coef']           = float(RL_KL_COEF)
cfg['rl']['vf_coef']           = float(RL_VF_COEF)
cfg['rl']['gamma']             = float(RL_GAMMA)
cfg['rl']['grad_clip']         = float(RL_GRAD_CLIP)
cfg['model']['freeze_clip']          = FREEZE_CLIP
cfg['model']['unfreeze_clip_vision'] = UNFREEZE_CLIP_VISION
cfg['training']['seed']              = SEED
cfg['training']['output_dir']        = RL_OUT_DIR

# Resolve "auto" device — PyTorch doesn't accept "auto" as a device string
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['training']['device'] = device_str
print(f'Device: {device_str} ({"GPU ✓" if device_str == "cuda" else "CPU — switch to T4 GPU in Runtime menu!"})')

if LT_DATA_PATH and os.path.isdir(LT_DATA_PATH):
    cfg['data']['episodes_path'] = LT_DATA_PATH
    cfg['data']['dataset_type']  = 'language_table'
    print(f'Data: Language-Table at {LT_DATA_PATH}')
else:
    cfg['data']['episodes_path'] = None
    cfg['data']['dataset_type']  = 'pkl'
    print('⚠️  No LT data — using synthetic (SR meaningless without real data)')

RL_CONFIG_PATH = f'{RL_OUT_DIR}/rl_config.yaml'
with open(RL_CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print(f'\nConfig → {RL_CONFIG_PATH}')
print('RL settings:')
for k, v in cfg['rl'].items():
    print(f'  {k}: {v}')

In [ ]:
# ── Cell 6: Run RL training ────────────────────────────────────────────────
# Output streams live — each epoch prints:
#   RL Epoch  N | steps XXXXX | return X.XXXX | success XX.X% | ...
#
# What to watch:
#   success XX.X%  → task success rate (want this to rise above SFT baseline)
#   entropy X.XXXX → keep > 0.5 (if near 0, raise RL_ENTROPY_COEF to 0.08)
#   kl X.XXXX      → keep < 0.5 (if > 1.0, raise RL_KL_COEF to 0.4)
#
# Best checkpoint → {RL_OUT_DIR}/best_rl_vera.pt  (auto-saved every epoch)

import subprocess, sys, os

cmd = [
    sys.executable, '-m', 'training.rl_trainer_vera',
    '--config', RL_CONFIG_PATH,
]

print(f'Starting RL training (seed={SEED}, {RL_EPOCHS} epochs × {RL_NUM_ROLLOUTS} rollouts)')
print(f'SFT checkpoint: {SFT_CKPT}')
print(f'Output dir:     {RL_OUT_DIR}')
print('─' * 70)

proc = subprocess.Popen(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\n' + '─' * 70)
    print('✓ RL training complete!')
    print(f'  Best checkpoint: {RL_OUT_DIR}/best_rl_vera.pt')
    print(f'  Full log:        {RL_OUT_DIR}/rl_vera_log.json')
    print(f'  CSV for plots:   {RL_OUT_DIR}/sample_efficiency.csv')
else:
    print(f'\n✗ RL training exited with code {proc.returncode}')
    print('Scroll up for the error.')

In [ ]:
# ── Cell 7: Plot results ───────────────────────────────────────────────────
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

LOG_PATH = f'{OUT_DIR}/rl/rl_vera_log.json'

if not os.path.exists(LOG_PATH):
    print(f'Log not found at {LOG_PATH} — run Cell 6 first.')
else:
    with open(LOG_PATH) as f:
        log = json.load(f)

    epochs   = [r['epoch']        for r in log]
    sr       = [r['success_rate'] * 100 for r in log]
    ret      = [r['mean_return']  for r in log]
    entropy  = [r['entropy']      for r in log]
    kl       = [r['kl_loss']      for r in log]
    steps    = [r['cumulative_steps'] for r in log]

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle(f'VERA RL Training — seed {SEED}', fontsize=13, fontweight='bold')

    # ── Success Rate ──
    ax = axes[0, 0]
    ax.plot(epochs, sr, color='#2563eb', linewidth=1.8)
    ax.axhline(10, color='#9ca3af', linewidth=1, linestyle='--', label='SFT baseline (~10%)')
    ax.fill_between(epochs, sr, 10, where=[s > 10 for s in sr],
                    color='#2563eb', alpha=0.12)
    ax.set_title('Task Success Rate (%)', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Success Rate (%)')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.grid(True, alpha=0.3)

    # ── Mean Return ──
    ax = axes[0, 1]
    ax.plot(epochs, ret, color='#16a34a', linewidth=1.8)
    ax.set_title('Mean Episode Return', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Return')
    ax.grid(True, alpha=0.3)

    # ── Entropy ──
    ax = axes[1, 0]
    ax.plot(epochs, entropy, color='#d97706', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Collapse risk < 0.5')
    ax.set_title('Policy Entropy (exploration)', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Entropy')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── KL vs BC ──
    ax = axes[1, 1]
    ax.plot(epochs, kl, color='#7c3aed', linewidth=1.8)
    ax.axhline(0.5, color='#ef4444', linewidth=1, linestyle='--', label='Forgetting risk > 0.5')
    ax.set_title('KL Divergence vs SFT Anchor', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('KL Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_PATH = f'{OUT_DIR}/rl/training_curves.png'
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved to {PLOT_PATH}')

    best_sr  = max(sr)
    best_ep  = epochs[sr.index(best_sr)]
    final_sr = sr[-1]
    print(f'\nSummary (seed {SEED}):')
    print(f'  SFT baseline:  ~10%')
    print(f'  Best RL SR:    {best_sr:.1f}% (epoch {best_ep})')
    print(f'  Final RL SR:   {final_sr:.1f}% (epoch {epochs[-1]})')
    print(f'  Gain over SFT: +{best_sr - 10:.1f} pp')

## Troubleshooting

| Symptom | Fix |
|---|---|
| `FileNotFoundError: best_sft_vera.pt` | Upload seed123's SFT checkpoint to `VERA_LT_Checkpoints/rl_seed123/best_sft_vera.pt` |
| Success rate stays at 0% for 30+ epochs | Raise `RL_ENTROPY_COEF` to `0.08` in Cell 4, re-run Cell 5+6 |
| Entropy collapses to ~0 | Same fix — more entropy bonus |
| KL explodes > 1.0 | Raise `RL_KL_COEF` to `0.4` |
| CUDA out of memory | Lower `RL_NUM_ROLLOUTS` to `4` |
| Runtime disconnects | Checkpoint auto-saved every epoch — re-run Cell 6, training restarts from scratch but best_rl_vera.pt is safe on Drive |
| `ModuleNotFoundError: language_table` | Cell 3 didn't finish — re-run it |

## Running all 3 seeds in parallel
1. Open 3 separate Colab tabs (File → Open in new tab)
2. In each tab, change `SEED` in Cell 4 to `42`, `123`, `456`
3. Make sure each has its own SFT checkpoint:
   - `rl_seed42/best_sft_vera.pt` (from your seed42 SFT run)
   - `rl_seed123/best_sft_vera.pt` ← **start here first**
   - `rl_seed456/best_sft_vera.pt`